# Notebook 7: Consolidated Biometric Evaluation - Grouped Electrode Categories & Combined Full Montage
## All 8 Models (4 ML + 4 DL), 3 Ensembles, Across Within-Session, Cross-Session, and Multi-Session LOSO Protocols

This master notebook implements the complete biometric authentication evaluation with **strictly spectral-domain and frequency-band power features (20 spectral features per electrode)**:
- **Electrode Groupings (Brain Region Categories):**
  1. **Frontal** (14 electrodes: Fp1, Fp2, AF3, AF4, F7, F5, F3, F1, Fz, F2, F4, F6, F8, FCz)
  2. **Central** (14 electrodes: FC5, FC3, FC1, FC2, FC4, FC6, T7, C5, C3, C1, Cz, C2, C4, C6)
  3. **Temporal** (6 electrodes: T8, TP7, CP5, CP6, TP8, FT7)
  4. **Parietal** (14 electrodes: CP3, CP1, CPz, CP2, CP4, P7, P5, P3, P1, Pz, P2, P4, P6, P8)
  5. **Occipital** (10 electrodes: PO7, PO5, PO3, POz, PO4, PO6, PO8, O1, Oz, O2)
- **Combined Electrodes (Full Montage):**
  6. **ALL** (All 58 electrodes combined into a 1,160-dimensional spectral vector)
- **Ensemble Architectures:**
  - **Soft Voting (Top-4)**: Mean posterior probability across SVM-RBF, Res-MLP, ExtraTrees, LightGBM.
  - **Rank Averaging (Top-4)**: Calibrated rank percentile fusion across the top 4 models.
  - **Stacker (Logistic Regression)**: Out-of-fold trained meta-classifier learning optimal model weights.
  - **Spatial Expert Fusion**: Multi-expert decision fusion across Frontal, Central, Parietal, and Occipital regional models.
- **Biometric Protocols:**
  1. **Within-Session (WS)**: 80:20 chronological genuine train/test split with balanced impostor set.
  2. **Cross-Session (CS)**: Train on 1 session, test on remaining 3 sessions (4 rotation folds).
  3. **Multi-Session LOSO**: Train on 3 sessions, test on the held-out 4th session (4 rotation folds).
- **All Results Saved**: Complete tabular display in notebook cells and exported to CSV files in `results/`.


In [1]:
# Environment setup, reproducible random seeds, and imports
import os
import sys
import time
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import signal
import mne
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier
import xgboost as xgb
import lightgbm as lgb

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    _ = torch.zeros(1, device=DEVICE)

print(f"Compute Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Random Seed   : {RANDOM_SEED}")


Compute Device: cuda (NVIDIA GeForce RTX 3050 Laptop GPU)
Random Seed   : 42


## Section 1: Strict Spectral-Domain Feature Extraction

In [2]:
# High-performance vectorized spectral feature extractor
_trapezoid = getattr(np, "trapezoid", getattr(np, "trapz", None))

FREQ_BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta":  (13.0, 30.0),
    "gamma": (30.0, 50.0)
}

def extract_spectral_features_tensor(data, sfreq=250.0):
    """Vectorized extraction of 20 spectral features per electrode channel."""
    n_epochs, n_channels, n_times = data.shape
    freqs, psd = signal.welch(data, fs=sfreq, nperseg=min(256, n_times), axis=-1)
    
    # 1. Absolute band powers (5)
    bp_list = []
    for (f_lo, f_hi) in FREQ_BANDS.values():
        mask = (freqs >= f_lo) & (freqs <= f_hi)
        p = _trapezoid(psd[:, :, mask], freqs[mask], axis=-1)
        bp_list.append(p)
    bp_arr = np.stack(bp_list, axis=-1)
    
    # 2. Total power (1-50 Hz) (1)
    tot_mask = (freqs >= 1.0) & (freqs <= 50.0)
    total_power = np.expand_dims(_trapezoid(psd[:, :, tot_mask], freqs[tot_mask], axis=-1), axis=-1) + 1e-12
    
    # 3. Relative band powers (5)
    rel_bp = bp_arr / total_power
    
    # 4. Spectral band power ratios (4)
    delta = bp_arr[..., 0] + 1e-12
    theta = bp_arr[..., 1] + 1e-12
    alpha = bp_arr[..., 2] + 1e-12
    beta  = bp_arr[..., 3] + 1e-12
    tbr = np.expand_dims(theta / beta, -1)
    tar = np.expand_dims(theta / alpha, -1)
    abr = np.expand_dims(alpha / beta, -1)
    slow_fast = np.expand_dims((delta + theta) / (alpha + beta), -1)
    ratios = np.concatenate([tbr, tar, abr, slow_fast], axis=-1)
    
    # 5. Spectral distribution & shape metrics (5)
    freqs_sub = freqs[tot_mask]
    psd_sub = psd[:, :, tot_mask]
    p_sum = psd_sub.sum(axis=-1, keepdims=True) + 1e-12
    centroid = (psd_sub * freqs_sub).sum(axis=-1, keepdims=True) / p_sum
    spread = np.sqrt(((freqs_sub - centroid)**2 * psd_sub).sum(axis=-1, keepdims=True) / p_sum)
    
    cum_psd = np.cumsum(psd_sub, axis=-1)
    cum_norm = cum_psd / (cum_psd[..., -1:] + 1e-12)
    idx50 = (cum_norm >= 0.5).argmax(axis=-1, keepdims=True)
    idx95 = (cum_norm >= 0.95).argmax(axis=-1, keepdims=True)
    sef50 = np.take_along_axis(np.broadcast_to(freqs_sub, idx50.shape[:-1] + (len(freqs_sub),)), idx50, axis=-1)
    sef95 = np.take_along_axis(np.broadcast_to(freqs_sub, idx95.shape[:-1] + (len(freqs_sub),)), idx95, axis=-1)
    
    idx_peak = psd_sub.argmax(axis=-1, keepdims=True)
    peak_freq = np.take_along_axis(np.broadcast_to(freqs_sub, idx_peak.shape[:-1] + (len(freqs_sub),)), idx_peak, axis=-1)
    spectral_dist = np.concatenate([centroid, spread, sef50, sef95, peak_freq], axis=-1)
    
    features = np.concatenate([bp_arr, total_power, rel_bp, ratios, spectral_dist], axis=-1)
    return features.astype(np.float32)

dataset_path = Path("../preprocessed_dataset")
if not dataset_path.exists():
    dataset_path = Path("preprocessed_dataset")

fif_files = sorted(list(dataset_path.glob("**/*_desc-preproc_epo.fif")))
EPOCH_CAP = 45

sample_ep = mne.read_epochs(str(fif_files[0]), preload=False, verbose=False)
ALL_CHANNELS = list(sample_ep.ch_names)
CHANNEL_TO_IDX = {ch: i for i, ch in enumerate(ALL_CHANNELS)}

REGIONS = {
    "Frontal": ["Fp1", "Fpz", "Fp2", "AF3", "AF4", "F7", "F5", "F3", "F1", "Fz", "F2", "F4", "F6", "F8"],
    "Central": ["FC5", "FC3", "FC1", "FCz", "FC2", "FC4", "FC6", "C5", "C3", "C1", "Cz", "C2", "C4", "C6"],
    "Temporal": ["T7", "T8", "TP7", "TP8", "CP5", "CP6"],
    "Parietal": ["CP3", "CP1", "CPz", "CP2", "CP4", "P7", "P5", "P3", "P1", "Pz", "P2", "P4", "P6", "P8"],
    "Occipital": ["PO7", "PO5", "PO3", "POz", "PO4", "PO6", "PO8", "O1", "Oz", "O2"],
    "ALL": ALL_CHANNELS
}

print(f"Extracting spectral features for all {len(fif_files)} files (Capped at {EPOCH_CAP} epochs)...")
t0 = time.time()
session_data = {}

for fpath in fif_files:
    fname = fpath.name
    parts = fpath.parts
    sub = [p for p in parts if p.startswith("sub-")][0]
    ses = [p for p in parts if p.startswith("ses-")][0]
    
    ep = mne.read_epochs(str(fpath), preload=True, verbose=False)
    raw_data = ep.get_data()[:EPOCH_CAP]
    sfreq = ep.info['sfreq']
    
    feats = extract_spectral_features_tensor(raw_data, sfreq=sfreq)
    
    key = (sub, ses)
    if key not in session_data:
        session_data[key] = []
    session_data[key].append(feats)

session_cache = {}
for (sub, ses), feat_list in session_data.items():
    session_cache[(sub, ses)] = np.vstack(feat_list)

SUBJECTS = sorted(list(set(sub for (sub, ses) in session_cache.keys())))
SUBJECT_SESSIONS = {
    sub: sorted([ses for (s, ses) in session_cache.keys() if s == sub])
    for sub in SUBJECTS
}

print(f"Spectral features ready in {time.time() - t0:.2f} seconds.")
print(f"Total Subjects: {len(SUBJECTS)} | Sessions per subject: 4 | Samples per session: 90")


Extracting spectral features for all 40 files (Capped at 45 epochs)...


Spectral features ready in 8.21 seconds.
Total Subjects: 5 | Sessions per subject: 4 | Samples per session: 90


## Section 2: Advanced Model Suite (4 ML + 4 DL)

In [3]:
# Biometric Metric Calculator
def evaluate_biometric_predictions(y_true, y_scores, threshold=0.5):
    """Compute standard biometric verification metrics: EER, FAR, FRR, FP, FN, Accuracy."""
    y_true = np.asarray(y_true).astype(int)
    y_scores = np.asarray(y_scores).astype(float)
    
    n_gen = int(np.sum(y_true == 1))
    n_imp = int(np.sum(y_true == 0))
    
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    fnr = 1.0 - tpr
    eer_idx = np.nanargmin(np.abs(fpr - fnr))
    eer = float((fpr[eer_idx] + fnr[eer_idx]) / 2.0)
    
    y_pred = (y_scores >= threshold).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    
    accuracy = float((tp + tn) / (n_gen + n_imp)) if (n_gen + n_imp) > 0 else 0.0
    far = float(fp / n_imp) if n_imp > 0 else 0.0
    frr = float(fn / n_gen) if n_gen > 0 else 0.0
    
    return {
        "Accuracy": accuracy,
        "EER": eer,
        "FAR": far,
        "FRR": frr,
        "FP": fp,
        "FN": fn
    }

# Deep Learning Architectures
class ResMLPNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, 64)
        self.relu = nn.ReLU()
        self.res1 = nn.Linear(64, 64)
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(64, 1)
    def forward(self, x):
        h = self.relu(self.fc1(x))
        h = h + self.drop(self.relu(self.res1(h)))
        return self.head(h).view(-1)

class CNN1DNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(4),
            nn.Flatten(),
            nn.Linear(32 * 4, 1)
        )
    def forward(self, x):
        x = x.unsqueeze(1)
        return self.net(x).view(-1)

class BiLSTMAttentionNet(nn.Module):
    def __init__(self, in_features, feat_dim=20):
        super().__init__()
        self.feat_dim = feat_dim
        dim = feat_dim if (in_features % feat_dim == 0 and in_features >= feat_dim) else in_features
        self.is_multichannel = (in_features % feat_dim == 0 and in_features >= feat_dim)
        self.lstm = nn.LSTM(dim if self.is_multichannel else 1, 16, batch_first=True, bidirectional=True)
        self.att = nn.Linear(32, 1)
        self.fc = nn.Linear(32, 1)
    def forward(self, x):
        if self.is_multichannel:
            x_seq = x.view(x.shape[0], -1, self.feat_dim)
        else:
            x_seq = x.unsqueeze(-1)
        out, _ = self.lstm(x_seq)
        weights = torch.softmax(self.att(out), dim=1)
        ctx = torch.sum(out * weights, dim=1)
        return self.fc(ctx).view(-1)

class SpecFormerNet(nn.Module):
    def __init__(self, in_features, d_model=24, nhead=3):
        super().__init__()
        if in_features % 20 == 0 and in_features >= 20:
            self.tok_proj = nn.Linear(20, d_model)
            self.multi_tok = True
        else:
            self.tok_proj = nn.Linear(1, d_model)
            self.multi_tok = False
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=48, batch_first=True, dropout=0.1)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.fc = nn.Linear(d_model, 1)
    def forward(self, x):
        if self.multi_tok:
            x_seq = x.view(x.shape[0], -1, 20)
        else:
            x_seq = x.unsqueeze(-1)
        h = self.tok_proj(x_seq)
        enc = self.encoder(h)
        return self.fc(enc.mean(dim=1)).view(-1)

class PyTorchClassifierWrapper:
    def __init__(self, net_cls, epochs=15, lr=3e-3, random_state=42):
        self.net_cls = net_cls
        self.epochs = epochs
        self.lr = lr
        self.random_state = random_state
        self.model = None

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        in_dim = X.shape[1]
        self.model = self.net_cls(in_dim).to(DEVICE)
        self.model.train()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        criterion = nn.BCEWithLogitsLoss()
        
        X_t = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        y_t = torch.tensor(y, dtype=torch.float32, device=DEVICE)
        
        for _ in range(self.epochs):
            optimizer.zero_grad()
            out = self.model(X_t)
            loss = criterion(out, y_t)
            loss.backward()
            optimizer.step()
        return self

    def predict_proba(self, X):
        self.model.eval()
        X_t = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            logits = self.model(X_t)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
        return np.column_stack([1.0 - probs, probs])

def get_model_pipeline(model_name, in_features):
    if model_name == "XGBoost":
        return xgb.XGBClassifier(
            n_estimators=60, max_depth=4, learning_rate=0.1,
            eval_metric="logloss", random_state=RANDOM_SEED, n_jobs=-1
        )
    elif model_name == "LightGBM":
        return lgb.LGBMClassifier(
            n_estimators=60, max_depth=4, num_leaves=15, learning_rate=0.1,
            random_state=RANDOM_SEED, n_jobs=-1, verbose=-1
        )
    elif model_name == "SVM_RBF":
        return SVC(C=10.0, kernel="rbf", gamma="scale", probability=True, random_state=RANDOM_SEED)
    elif model_name == "ExtraTrees":
        return ExtraTreesClassifier(n_estimators=60, max_depth=None, random_state=RANDOM_SEED, n_jobs=-1)
    elif model_name == "Res-MLP":
        return PyTorchClassifierWrapper(ResMLPNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    elif model_name == "1D-CNN":
        return PyTorchClassifierWrapper(CNN1DNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    elif model_name == "BiLSTM-Attention":
        return PyTorchClassifierWrapper(BiLSTMAttentionNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    elif model_name == "SpecFormer":
        return PyTorchClassifierWrapper(SpecFormerNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    else:
        raise ValueError(f"Unknown model: {model_name}")

ALL_MODEL_NAMES = [
    "XGBoost", "LightGBM", "SVM_RBF", "ExtraTrees",
    "Res-MLP", "1D-CNN", "BiLSTM-Attention", "SpecFormer"
]
print("All 8 advanced model architectures verified and registered.")


All 8 advanced model architectures verified and registered.


## Section 3: Grouped Categories & Evaluation Framework

We group all electrodes by category (5 brain regions) and combined montage (ALL 58 channels), plus Spatial Expert Fusion.


In [4]:
import os
from pathlib import Path
from scipy import stats as sps
from sklearn.linear_model import LogisticRegression

# Create results directories
for rpath in [Path('results'), Path('eeg_biometrics_2/results')]:
    try:
        rpath.mkdir(parents=True, exist_ok=True)
    except Exception:
        pass

SUB_ARR = {s: np.vstack([session_cache[(s, ses)] for ses in SUBJECT_SESSIONS[s]]) for s in SUBJECTS}  # (360, 58, 20)
BASE4 = ['SVM_RBF', 'Res-MLP', 'ExtraTrees', 'LightGBM']
ENS_NAMES = ['Ens_SoftVote', 'Ens_RankAvg', 'Ens_Stacker']
METRICS = ['Accuracy', 'EER', 'FAR', 'FRR', 'FP', 'FN']
MODEL_TYPE = {**{m: 'ML' for m in ['XGBoost', 'LightGBM', 'SVM_RBF', 'ExtraTrees']},
              **{m: 'DL' for m in ['Res-MLP', '1D-CNN', 'BiLSTM-Attention', 'SpecFormer']},
              **{e: 'Ensemble' for e in ENS_NAMES}, 'Spatial_Fusion': 'Ensemble'}

def counts(n, k):
    b, r = divmod(n, k)
    return [b + (1 if i < r else 0) for i in range(k)]

def pick(pool, c):
    if c >= len(pool):
        return pool
    return pool[np.linspace(0, len(pool) - 1, c).round().astype(int)]

def make_split(D, target, kind, tr_idx, te_idx):
    others = [s for s in SUBJECTS if s != target]
    if kind == 'WS':
        g = D[target]; ntr = int(len(g) * 0.8)
        g_tr, g_te = g[:ntr], g[ntr:]
        pools_tr = [D[o][:int(len(D[o]) * 0.8)] for o in others]
        pools_te = [D[o][int(len(D[o]) * 0.8):] for o in others]
    else:
        ss = SUBJECT_SESSIONS[target]
        g_tr = np.vstack([D[(target, ss[i])] for i in tr_idx]); g_te = np.vstack([D[(target, ss[i])] for i in te_idx])
        pools_tr = [np.vstack([D[(o, SUBJECT_SESSIONS[o][i])] for i in tr_idx]) for o in others]
        pools_te = [np.vstack([D[(o, SUBJECT_SESSIONS[o][i])] for i in te_idx]) for o in others]
    i_tr = np.vstack([pick(p, c) for p, c in zip(pools_tr, counts(len(g_tr), len(others)))])
    i_te = np.vstack([pick(p, c) for p, c in zip(pools_te, counts(len(g_te), len(others)))])
    Xtr = np.vstack([g_tr, i_tr]); ytr = np.r_[np.ones(len(g_tr)), np.zeros(len(i_tr))]
    Xte = np.vstack([g_te, i_te]); yte = np.r_[np.ones(len(g_te)), np.zeros(len(i_te))]
    return Xtr, ytr, Xte, yte

def block_folds(y, k=2):
    fold = np.zeros(len(y), dtype=int)
    for c in (0, 1):
        idx = np.where(y == c)[0]
        fold[idx] = (np.arange(len(idx)) * k) // len(idx)
    return fold

def fit_all(Xtr, ytr, Xte, yte, want_weights=False):
    sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
    probs = {}
    for m in ALL_MODEL_NAMES:
        probs[m] = get_model_pipeline(m, Xtr.shape[1]).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    k_folds = 2
    folds = block_folds(ytr, k_folds)
    oof = {}
    for m in BASE4:
        o = np.zeros(len(ytr))
        for f in range(k_folds):
            tr, va = folds != f, folds == f
            o[va] = get_model_pipeline(m, Xtr.shape[1]).fit(Xtr[tr], ytr[tr]).predict_proba(Xtr[va])[:, 1]
        oof[m] = o
    out = dict(probs)
    out['Ens_SoftVote'] = np.mean([probs[m] for m in BASE4], axis=0)
    out['Ens_RankAvg'] = np.mean([np.searchsorted(np.sort(oof[m]), probs[m]) / len(ytr) for m in BASE4], axis=0)
    meta = LogisticRegression(C=1.0, random_state=RANDOM_SEED).fit(np.column_stack([oof[m] for m in BASE4]), ytr)
    out['Ens_Stacker'] = meta.predict_proba(np.column_stack([probs[m] for m in BASE4]))[:, 1]
    res = {k: evaluate_biometric_predictions(yte, v) for k, v in out.items()}
    return res, (meta.coef_[0] if want_weights else None), len(ytr), int(yte.sum()), int((1 - yte).sum())

def make_D(kind, chs):
    ci = [CHANNEL_TO_IDX[c] for c in chs]
    if kind == 'WS':
        return {s: SUB_ARR[s][:, ci, :].reshape(len(SUB_ARR[s]), -1) for s in SUBJECTS}
    return {k: v[:, ci, :].reshape(v.shape[0], -1) for k, v in session_cache.items()}

PROTOCOLS = {
    'WS':   [('80:20 chrono', 'first 80% of each user', 'last 20% of each user', None, None)],
    'CS':   [(f'train S{i+1}', f'Session {i+1}', 'Sessions ' + ','.join(str(j+1) for j in range(4) if j != i), [i], [j for j in range(4) if j != i]) for i in range(4)],
    'LOSO': [(f'hold-out S{i+1}', 'Sessions ' + ','.join(str(j+1) for j in range(4) if j != i), f'Session {i+1}', [j for j in range(4) if j != i], [i]) for i in range(4)],
}
PROTO_TITLE = {'WS': 'Within-session 80:20', 'CS': 'Cross-session (train 1 session, test other 3)', 'LOSO': 'Multi-session LOSO (train 3 sessions, test 1)'}

# Grouped categories and combined electrodes
CONFIGS = [
    ('Category', 'Frontal', 'Frontal (14 electrodes)', REGIONS['Frontal']),
    ('Category', 'Central', 'Central (14 electrodes)', REGIONS['Central']),
    ('Category', 'Temporal', 'Temporal (6 electrodes)', REGIONS['Temporal']),
    ('Category', 'Parietal', 'Parietal (14 electrodes)', REGIONS['Parietal']),
    ('Category', 'Occipital', 'Occipital (10 electrodes)', REGIONS['Occipital']),
    ('Combined', 'ALL', 'Combined All 58 Electrodes', REGIONS['ALL'])
]
SPATIAL = ['Frontal', 'Central', 'Parietal', 'Occipital']

RESULTS = []
STACK_W = []

def run_protocol(kind):
    t0 = time.time()
    print(f'=== Starting Protocol: {PROTO_TITLE[kind]} ===')
    for ci, (level, name, desc, chs) in enumerate(CONFIGS):
        D = make_D(kind, chs)
        for (fold, trl, tel, tr_idx, te_idx) in PROTOCOLS[kind]:
            per = {}; n_tr = n_g = n_i = 0
            for t in SUBJECTS:
                Xtr, ytr, Xte, yte = make_split(D, t, kind, tr_idx, te_idx)
                res, w, n_tr, n_g, n_i = fit_all(Xtr, ytr, Xte, yte, want_weights=(name == 'ALL'))
                for m, r in res.items():
                    per.setdefault(m, []).append(r)
                if w is not None:
                    STACK_W.append({'Protocol': kind, 'Fold': fold, **{m: float(x) for m, x in zip(BASE4, w)}})
            for m, lst in per.items():
                RESULTS.append({'Protocol': kind, 'Fold': fold, 'Train_Sessions': trl, 'Test_Sessions': tel,
                                'Level': level, 'Config': name, 'Description': desc, 'Model': m, 'Type': MODEL_TYPE[m],
                                'N_train': n_tr, 'N_genuine_test': n_g, 'N_impostor_test': n_i,
                                **{k: float(np.mean([x[k] for x in lst])) for k in METRICS}})
        print(f'[{kind}] Finished config {ci+1}/{len(CONFIGS)}: {name} ({time.time()-t0:.1f}s)')
        
    # Spatial-fusion ensemble (SVM-RBF per region, probabilities averaged)
    DR = {r: make_D(kind, REGIONS[r]) for r in SPATIAL}
    for (fold, trl, tel, tr_idx, te_idx) in PROTOCOLS[kind]:
        per = []; n_tr = n_g = n_i = 0
        for t in SUBJECTS:
            pr = []
            for r in SPATIAL:
                Xtr, ytr, Xte, yte = make_split(DR[r], t, kind, tr_idx, te_idx)
                sc = StandardScaler().fit(Xtr)
                pr.append(get_model_pipeline('SVM_RBF', Xtr.shape[1]).fit(sc.transform(Xtr), ytr).predict_proba(sc.transform(Xte))[:, 1])
            per.append(evaluate_biometric_predictions(yte, np.mean(pr, axis=0)))
            n_tr, n_g, n_i = len(ytr), int(yte.sum()), int((1 - yte).sum())
        RESULTS.append({'Protocol': kind, 'Fold': fold, 'Train_Sessions': trl, 'Test_Sessions': tel,
                        'Level': 'Ensemble_Fusion', 'Config': 'Spatial_Fusion', 'Description': 'Spatial Expert Fusion (F+C+P+O)', 
                        'Model': 'Spatial_Fusion', 'Type': 'Ensemble',
                        'N_train': n_tr, 'N_genuine_test': n_g, 'N_impostor_test': n_i,
                        **{k: float(np.mean([x[k] for x in per])) for k in METRICS}})
    print(f'[{kind}] All configurations completed in {time.time()-t0:.1f}s\n')

print(f'Configurations to evaluate: {len(CONFIGS)} (5 regional categories + 1 combined 58-channel montage + Spatial Fusion)')
print(f'Models: {len(ALL_MODEL_NAMES)} (4 ML + 4 DL) + {len(ENS_NAMES)} ensembles + Spatial Expert Fusion')


Configurations to evaluate: 6 (5 regional categories + 1 combined 58-channel montage + Spatial Fusion)
Models: 8 (4 ML + 4 DL) + 3 ensembles + Spatial Expert Fusion


## Section 4: Run Protocol 1 - Within-Session 80:20 (WS)

In [5]:
run_protocol('WS')

=== Starting Protocol: Within-session 80:20 ===


[WS] Finished config 1/6: Frontal (8.2s)


[WS] Finished config 2/6: Central (13.4s)


[WS] Finished config 3/6: Temporal (17.5s)


[WS] Finished config 4/6: Parietal (22.5s)


[WS] Finished config 5/6: Occipital (27.1s)


[WS] Finished config 6/6: ALL (40.9s)


[WS] All configurations completed in 41.7s



## Section 5: Run Protocol 2 - Cross-Session Single-Session Train (CS)

In [6]:
run_protocol('CS')

=== Starting Protocol: Cross-session (train 1 session, test other 3) ===


[CS] Finished config 1/6: Frontal (14.2s)


[CS] Finished config 2/6: Central (29.0s)


[CS] Finished config 3/6: Temporal (42.0s)


[CS] Finished config 4/6: Parietal (57.4s)


[CS] Finished config 5/6: Occipital (71.7s)


[CS] Finished config 6/6: ALL (99.4s)


[CS] All configurations completed in 101.4s



## Section 6: Run Protocol 3 - Multi-Session LOSO (Train 3, Test 1)

In [7]:
run_protocol('LOSO')
RES = pd.DataFrame(RESULTS)
print(f'Total result records generated: {len(RES)}')


=== Starting Protocol: Multi-session LOSO (train 3 sessions, test 1) ===


[LOSO] Finished config 1/6: Frontal (27.9s)


[LOSO] Finished config 2/6: Central (55.3s)


[LOSO] Finished config 3/6: Temporal (77.4s)


[LOSO] Finished config 4/6: Parietal (98.8s)


[LOSO] Finished config 5/6: Occipital (121.5s)


[LOSO] Finished config 6/6: ALL (178.9s)


[LOSO] All configurations completed in 183.8s

Total result records generated: 603


## Section 7: Complete Consolidated Tables and Saving Results

All tables are displayed below with complete precision and saved into CSV files inside `results/`.


In [8]:
MODEL_ORDER = ALL_MODEL_NAMES + ENS_NAMES + ['Spatial_Fusion']

def show(title, df, idx=False):
    print('\n' + '=' * 130)
    print(title)
    print('=' * 130)
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 250):
        print(df.round(4).to_string(index=idx))

# 1. Summary of Each Protocol: Category Groups & Combined Montage across all Models
for kind in ('WS', 'CS', 'LOSO'):
    d = RES[RES.Protocol == kind]
    
    # Fold design & sample counts
    sample_df = d[d.Config == 'ALL'][['Fold', 'Train_Sessions', 'Test_Sessions', 'N_train', 'N_genuine_test', 'N_impostor_test']].drop_duplicates()
    show(f'[{kind}] {PROTO_TITLE[kind]} - EXPERIMENTAL DESIGN & SAMPLE COUNTS', sample_df)
    
    # Mean performance over folds
    r = d.groupby(['Level', 'Config', 'Model', 'Type'])[METRICS].mean().reset_index()
    r['o'] = r.Model.map({m: i for i, m in enumerate(MODEL_ORDER)})
    r = r.sort_values(['Level', 'Config', 'o']).drop(columns='o')
    show(f'[{kind}] {PROTO_TITLE[kind]} - CATEGORY GROUPS & COMBINED MONTAGE x ALL MODELS (Mean Over Folds)', r)
    
    # Pivot matrix: Config vs Model EER
    pivot_eer = d.groupby(['Config', 'Model'])['EER'].mean().unstack()[MODEL_ORDER[:len(ALL_MODEL_NAMES) + len(ENS_NAMES)]]
    show(f'[{kind}] EER MATRIX: CONFIGURATIONS x MODELS', pivot_eer, idx=True)

# 2. Per-Session Fold Breakdown for Combined ALL Montage
for kind in ('CS', 'LOSO'):
    d = RES[(RES.Protocol == kind) & (RES.Config == 'ALL')]
    f = d[['Fold', 'Train_Sessions', 'Test_Sessions', 'Model', 'Type'] + METRICS].copy()
    f['o'] = f.Model.map({m: i for i, m in enumerate(MODEL_ORDER)})
    f = f.sort_values(['Fold', 'o']).drop(columns='o')
    show(f'[{kind}] PER-SESSION-FOLD RESULTS - COMBINED ALL MONTAGE', f)

# 3. Best Single Model vs Best Ensemble Comparison
rows = []
for kind in ('WS', 'CS', 'LOSO'):
    for cfg in ['Frontal', 'Central', 'Temporal', 'Parietal', 'Occipital', 'ALL']:
        dd = RES[(RES.Protocol == kind) & (RES.Config == cfg)]
        g = dd.groupby(['Model', 'Type'])[METRICS].mean().reset_index()
        bs = g[g.Type.isin(['ML', 'DL'])].sort_values('EER').iloc[0]
        be = g[g.Type == 'Ensemble'].sort_values('EER').iloc[0]
        rows.append({
            'Protocol': kind, 'Configuration': cfg,
            'Best_Single_Model': bs.Model, 'Single_EER': bs.EER, 'Single_Acc': bs.Accuracy,
            'Best_Ensemble': be.Model, 'Ensemble_EER': be.EER, 'Ensemble_Acc': be.Accuracy,
            'EER_Reduction_%': (bs.EER - be.EER) / bs.EER * 100,
            'Accuracy_Gain_%': (be.Accuracy - bs.Accuracy) * 100
        })
ens_comp_df = pd.DataFrame(rows)
show('BEST SINGLE MODEL vs BEST ENSEMBLE PER CONFIGURATION & PROTOCOL', ens_comp_df)

# 4. Overall Winner Configurations
win = []
for kind in ('WS', 'CS', 'LOSO'):
    d = RES[RES.Protocol == kind]
    for lvl in ('Category', 'Combined'):
        sub = d[d.Level == lvl]
        best_row = sub.loc[sub['EER'].idxmin()]
        win.append({
            'Protocol': kind, 'Level': lvl, 'Best_Config': best_row.Config,
            'Best_Model': best_row.Model, 'Type': best_row.Type,
            **{m: best_row[m] for m in METRICS}
        })
win_df = pd.DataFrame(win)
show('OVERALL WINNER CONFIGURATIONS AND MODELS', win_df)

# 5. Stacker Weights Summary
if STACK_W:
    sw = pd.DataFrame(STACK_W).groupby(['Protocol', 'Fold'])[BASE4].mean().reset_index()
    show('STACKER (LOGISTIC REGRESSION) LEARNED WEIGHTS PER BASE MODEL (ALL Montage)', sw)
    for p in [Path('results'), Path('eeg_biometrics_2/results')]:
        try:
            p.mkdir(parents=True, exist_ok=True)
            sw.to_csv(p / 'stacker_weights.csv', index=False)
        except Exception:
            pass

# 6. Save Complete Results CSVs
for p in [Path('results'), Path('eeg_biometrics_2/results')]:
    try:
        p.mkdir(parents=True, exist_ok=True)
        RES.to_csv(p / 'complete_results_grouped_and_combined.csv', index=False)
        ens_comp_df.to_csv(p / 'ensemble_vs_single_comparison.csv', index=False)
        win_df.to_csv(p / 'overall_winners_summary.csv', index=False)
    except Exception:
        pass

print('\n[SUCCESS] All evaluation results saved to CSV in directory: results/')



[WS] Within-session 80:20 - EXPERIMENTAL DESIGN & SAMPLE COUNTS
        Fold         Train_Sessions         Test_Sessions  N_train  N_genuine_test  N_impostor_test
80:20 chrono first 80% of each user last 20% of each user      576              72               72

[WS] Within-session 80:20 - CATEGORY GROUPS & COMBINED MONTAGE x ALL MODELS (Mean Over Folds)
          Level         Config            Model     Type  Accuracy    EER    FAR    FRR   FP   FN
       Category        Central          XGBoost       ML    0.9500 0.0486 0.0611 0.0389  4.4  2.8
       Category        Central         LightGBM       ML    0.9514 0.0556 0.0583 0.0389  4.2  2.8
       Category        Central          SVM_RBF       ML    0.9542 0.0333 0.0389 0.0528  2.8  3.8
       Category        Central       ExtraTrees       ML    0.9417 0.0528 0.0639 0.0528  4.6  3.8
       Category        Central          Res-MLP       DL    0.8903 0.0778 0.1056 0.1139  7.6  8.2
       Category        Central           1D-CNN     


OVERALL WINNER CONFIGURATIONS AND MODELS
Protocol    Level Best_Config   Best_Model     Type  Accuracy    EER    FAR    FRR   FP   FN
      WS Category     Central  Ens_Stacker Ensemble    0.9569 0.0292 0.0278 0.0583  2.0  4.2
      WS Combined         ALL  Ens_Stacker Ensemble    0.9611 0.0153 0.0083 0.0694  0.6  5.0
      CS Category     Central Ens_SoftVote Ensemble    0.9000 0.0863 0.0733 0.1267 19.8 34.2
      CS Combined         ALL  Ens_RankAvg Ensemble    0.9026 0.0730 0.1459 0.0489 39.4 13.2
    LOSO Category    Parietal  Ens_Stacker Ensemble    0.9611 0.0278 0.0156 0.0622  1.4  5.6
    LOSO Combined         ALL Ens_SoftVote Ensemble    0.9778 0.0133 0.0133 0.0311  1.2  2.8



STACKER (LOGISTIC REGRESSION) LEARNED WEIGHTS PER BASE MODEL (ALL Montage)
Protocol         Fold  SVM_RBF  Res-MLP  ExtraTrees  LightGBM
      CS     train S1   2.5738   0.3636      0.9175    0.7108
      CS     train S2   1.8150   0.8060      0.3396   -0.0383
      CS     train S3   0.9252   0.7535      0.2859   -0.3375
      CS     train S4   1.7629  -0.0272     -0.3052    0.9223
    LOSO  hold-out S1   3.1549   0.3303      1.8916    0.3077
    LOSO  hold-out S2   3.5770   0.1847      1.6552   -0.2034
    LOSO  hold-out S3   4.0134  -0.0092      1.2368    0.4401
    LOSO  hold-out S4   4.1907   0.3389      1.7382   -0.0283
      WS 80:20 chrono   3.6664   0.5158      1.4829    0.4201

[SUCCESS] All evaluation results saved to CSV in directory: results/
